# Stage 2 — Local Track

Same model and data as stage 1. What changes: training runs in a container
and every run is recorded in mlflow instead of overwriting `runs/local-train/`.

## Environment

Run this inside the compose stack — `docker compose up -d`, then open
Jupyter at http://127.0.0.1:8888. `MLFLOW_TRACKING_URI` is set by compose.

In [1]:
import sys
from pathlib import Path

import mlflow
import torch
import ultralytics

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

from src.tracking import tracking_uri

print("python     ", sys.version.split()[0])
print("torch      ", torch.__version__)
print("ultralytics", ultralytics.__version__)
print("mlflow     ", mlflow.__version__)
print("cuda       ", torch.cuda.is_available())
print("tracking   ", tracking_uri())

python      3.12.13
torch       2.13.0+cu130
ultralytics 8.4.116
mlflow      3.15.1
cuda        False
tracking    http://mlflow:5000


In [2]:
# Fail here rather than 20 minutes into training if the server is unreachable.
mlflow.set_tracking_uri(tracking_uri())
print(mlflow.search_experiments())

[<Experiment: artifact_location='/mlflow/artifacts/2', creation_time=1786163766119, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1786163766119, lifecycle_stage='active', name='yolo-plate-detection', tags={}, trace_location=None, workspace='default'>, <Experiment: artifact_location='/mlflow/artifacts/0', creation_time=1786162182159, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1786162182159, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]


## Data

Rebuild the split and descriptor. `data.yaml` holds an absolute path, so it
must be regenerated inside the container — the host copy points at a Windows
path that does not exist here.

`LIMIT = None` uses all 556 pairs. Stage 1's container run used 200, which is
why its metrics were lower.

In [3]:
from src.data_loader import build_split, summarize, verify_split, write_data_yaml

LIMIT = None

stats = summarize(RAW)
print({k: stats[k] for k in ("pairs", "boxes_total", "boxes_per_image_max", "malformed")})

print(build_split(RAW, PROCESSED, val_fraction=0.2, limit=LIMIT, seed=0))
print(verify_split(PROCESSED))

names = (RAW / "classes.txt").read_text().split()
data_yaml = write_data_yaml(ROOT / "configs" / "data.yaml", PROCESSED, names)
print(data_yaml.read_text())

{'pairs': 556, 'boxes_total': 574, 'boxes_per_image_max': 3, 'malformed': []}
{'train': 445, 'val': 111, 'orphan_images': 0, 'orphan_labels': 0}
{'train': 445, 'val': 111}
path: /workspace/data/processed
train: train/images
val: val/images
nc: 1
names: ['car_plate']



## Train with tracking

Ultralytics ships an mlflow callback, enabled by default, that reads
`MLFLOW_TRACKING_URI` and logs params, per-epoch metrics and everything in
`save_dir` — including `best.pt`. No logging code is needed around
`model.train()`.

`MLFLOW_EXPERIMENT_NAME` and `MLFLOW_RUN` name the experiment and run;
otherwise they default to the ultralytics `project` and `name`.

In [4]:
import os
import time

import yaml
from ultralytics import YOLO

EXPERIMENT = "yolo-plate-detection"

train_cfg = yaml.safe_load((ROOT / "configs" / "train.yaml").read_text())
cfg = dict(train_cfg)
model_weights = cfg.pop("model")
cfg["project"] = str(ROOT / cfg["project"])

n_train = len(list((PROCESSED / "train" / "images").iterdir()))
os.environ["MLFLOW_EXPERIMENT_NAME"] = EXPERIMENT
os.environ["MLFLOW_RUN"] = f"cpu-{n_train}img-{cfg['epochs']}ep-{cfg['imgsz']}px"

# Keep the run open after training so dataset provenance can be appended.
os.environ["MLFLOW_KEEP_RUN_ACTIVE"] = "true"

print(f"experiment {EXPERIMENT}")
print(f"run        {os.environ['MLFLOW_RUN']}")

model = YOLO(model_weights)
start = time.time()
results = model.train(data=str(data_yaml), **cfg)
print(f"\nelapsed: {time.time() - start:.0f}s")

experiment yolo-plate-detection
run        cpu-445img-10ep-416px
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.13.0+cu130 CPU (Intel Core 5 120U)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspace/configs/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosai

2026/08/08 04:38:52 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/08/08 04:38:52 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably

MLflow: logging run_id(00664d2dcf414d6f93668a01aecfba33) to http://mlflow:5000
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 416 train, 416 val
Using 0 dataloader workers
Logging results to /workspace/runs/local-train
Starting training for 10 epochs...
Closing dataloader mosaic

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/10         0G       1.11      2.636       1.02          6        416: 100% ━━━━━━━━━━━━ 56/56 1.7s/it 1:381.3sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.6s/it 11.0s.8ss
                   all        111        116    0.00335      0.207     0.0703     0.0359

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/10         0G       1.09      1.612      1.002          5        416: 100% ━━━━━━━━━━━━ 56/56 1.5s/it 1:271.3sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50

### Dataset provenance

The callback logs `trainer.args` but not how many images the split held. Two
runs with identical hyperparameters and different dataset sizes would be
indistinguishable, which is exactly what happened across stage 1.

In [5]:
from src.tracking import log_dataset_context

logged = log_dataset_context(
    PROCESSED,
    RAW,
    **{"data.limit": str(LIMIT), "data.seed": 0, "run.device": "cpu"},
)

run = mlflow.active_run()
print(f"run_id {run.info.run_id}")
for key, value in logged.items():
    print(f"  {key:26} {value}")

mlflow.end_run()
print("\nrun closed")

run_id 00664d2dcf414d6f93668a01aecfba33
  data.train_images          445
  data.train_boxes           458
  data.val_images            111
  data.val_boxes             116
  data.total_images          556
  data.available_images      556
  data.fraction_used         1.0
  data.limit                 None
  data.seed                  0
  run.device                 cpu
🏃 View run cpu-445img-10ep-416px at: http://mlflow:5000/#/experiments/2/runs/00664d2dcf414d6f93668a01aecfba33
🧪 View experiment at: http://mlflow:5000/#/experiments/2

run closed


## Verify what was logged

Read the run back from the tracking server rather than trusting that logging
worked.

In [6]:
from src.tracking import latest_run_id

run_id = latest_run_id(EXPERIMENT)
fetched = mlflow.get_run(run_id)

print(f"run_id  {run_id}")
print(f"status  {fetched.info.status}")

print("\nfinal metrics")
for key in sorted(fetched.data.metrics):
    if "mAP" in key or "precision" in key or "recall" in key:
        print(f"  {key:28} {fetched.data.metrics[key]:.4f}")

print("\ndataset params")
for key in sorted(k for k in fetched.data.params if k.startswith("data.")):
    print(f"  {key:28} {fetched.data.params[key]}")

print("\nartifacts")
for artifact in mlflow.artifacts.list_artifacts(run_id=run_id):
    print(f"  {artifact.path}")

run_id  00664d2dcf414d6f93668a01aecfba33
status  FINISHED

final metrics
  metrics/mAP50-95B            0.7494
  metrics/mAP50B               0.9502
  metrics/precisionB           0.9616
  metrics/recallB              0.9310

dataset params
  data.available_images        556
  data.fraction_used           1.0
  data.limit                   None
  data.seed                    0
  data.total_images            556
  data.train_boxes             458
  data.train_images            445
  data.val_boxes               116
  data.val_images              111

artifacts
  BoxF1_curve.png
  BoxPR_curve.png
  BoxP_curve.png
  BoxR_curve.png
  args.yaml
  confusion_matrix.png
  confusion_matrix_normalized.png
  labels.jpg
  results.csv
  results.png
  train_batch0.jpg
  train_batch1.jpg
  train_batch2.jpg
  val_batch0_labels.jpg
  val_batch0_pred.jpg
  val_batch1_labels.jpg
  val_batch1_pred.jpg
  val_batch2_labels.jpg
  val_batch2_pred.jpg
  weights


In [7]:
# Per-epoch history -- the callback logs a point per epoch, so convergence is
# visible without re-reading results.csv.
#
# Note the key: the callback strips parentheses, so it is "metrics/mAP50-95B"
# here, not the "metrics/mAP50-95(B)" that ultralytics uses in results.csv.
client = mlflow.tracking.MlflowClient()
history = client.get_metric_history(run_id, "metrics/mAP50-95B")

print(f"{'epoch':>6} {'mAP50-95':>10}")
for point in history:
    print(f"{point.step:>6} {point.value:>10.4f}")

 epoch   mAP50-95
     0     0.0359
     1     0.4264
     2     0.5039
     3     0.6316
     4     0.6581
     5     0.6751
     6     0.7173
     7     0.7266
     8     0.7446
     9     0.7492
    10     0.7494


## Compare runs

The point of the stage. Every run in the experiment, side by side.

In [8]:
from src.tracking import compare_runs

compare_runs(EXPERIMENT)

,mlflow.runName,epochs,imgsz,data.train_images,metrics/mAP50B,metrics/mAP50-95B,metrics/precisionB,metrics/recallB
0,cpu-445img-10ep-416px,10,416,445,0.950199,0.749393,0.961617,0.931034
1,cpu-445img-10ep-416px,10,416,None,NaN,NaN,NaN,NaN


Browse the same data at http://127.0.0.1:5000.